# WikiText-2 LSTM language model

A word-level model trained from scratch. It estimates the probability of the next word given the words so far, then uses that distribution to continue a phrase.

Run this notebook from the project directory. A full training run is 8 epochs and takes about 20 minutes on an Apple GPU.


In [ ]:
import math
import re
from collections import Counter
from pathlib import Path

import torch
from torch import nn

UNK = "<unk>"
EOS = "<eos>"
MIN_FREQ = 3
BATCH_SIZE = 20
SEQ_LEN = 35
EPOCHS = 8
CLIP = 1.0
LEARNING_RATE = 1e-3
SEED = 1111

ROOT = Path.cwd()
DATA_PATH = ROOT / "data" / f"wikitext2_min{MIN_FREQ}.pt"
CHECKPOINT_PATH = ROOT / "checkpoints" / "lstm.pt"

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print(device)


## Text and vocabulary

WikiText-2 raw is already split on spaces. `@-@`, `@,@`, and `@.@` are punctuation placeholders, so they are turned back into `-`, `,`, and `.` before splitting. Blank lines and article headings (`= Title =`) are dropped. Each remaining line ends with `<eos>`.

The vocabulary is counted on the training split only. A word kept fewer than 3 times, and any validation or test word that was never kept, becomes `<unk>`.


In [ ]:
PLACEHOLDERS = (("@-@", "-"), ("@,@", ","), ("@.@", "."))
SPACE_BEFORE_PUNCT = re.compile(r"\s+([.,;:!?%)\]}])")
SPACE_AFTER_OPEN = re.compile(r"([(\[{])\s+")
CONTRACTION = re.compile(r"\s+((?:n't)|(?:'(?:s|re|ve|ll|d|m|t))\b)", re.IGNORECASE)


def normalize(text):
    for placeholder, symbol in PLACEHOLDERS:
        text = text.replace(placeholder, symbol)
    return text


def is_heading(line):
    stripped = line.strip()
    return bool(stripped) and stripped.startswith("=") and stripped.endswith("=")


def tokenize_line(line):
    return normalize(line).split()


def tokenize_prompt(text):
    # No <eos> here: generation should continue the sentence, not start a new line.
    text = normalize(text.strip())
    text = text.replace("n't", " n't")
    text = re.sub(r"(?i)(?<!n)'(s|re|ve|ll|d|m|t)\b", r" '\1", text)
    text = re.sub(r"([^\w\s'])", r" \1 ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text.split()


def tokens_from_rows(rows):
    tokens = []
    for row in rows:
        line = row.strip()
        if not line or is_heading(line):
            continue
        words = tokenize_line(line)
        if not words:
            continue
        tokens.extend(words)
        tokens.append(EOS)
    return tokens


def detokenize(tokens):
    pieces = ["\n" if token == EOS else token for token in tokens]
    text = " ".join(pieces)
    text = CONTRACTION.sub(r"\1", text)
    text = re.sub(r"\s+-\s+", "-", text)
    text = SPACE_BEFORE_PUNCT.sub(r"\1", text)
    text = SPACE_AFTER_OPEN.sub(r"\1", text)
    return text.replace(" \n", "\n").replace("\n ", "\n").strip()


class Vocabulary:
    def __init__(self, itos):
        self.itos = list(itos)
        self.stoi = {word: index for index, word in enumerate(self.itos)}

    def __len__(self):
        return len(self.itos)

    @property
    def unk_id(self):
        return self.stoi[UNK]

    def encode(self, tokens):
        unk = self.unk_id
        return [self.stoi.get(token, unk) for token in tokens]

    def decode(self, ids):
        return detokenize(self.itos[index] for index in ids)


def build_vocab(train_tokens, min_freq=MIN_FREQ):
    counts = Counter(train_tokens)
    itos = [UNK, EOS]
    for word, count in counts.most_common():
        if word in (UNK, EOS):
            continue
        if count >= min_freq:
            itos.append(word)
    return Vocabulary(itos)


In [ ]:
def load_corpus(min_freq=MIN_FREQ):
    if DATA_PATH.exists():
        saved = torch.load(DATA_PATH, weights_only=False)
        saved["vocab"] = Vocabulary(saved["itos"])
        return saved

    from datasets import load_dataset

    dataset = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1")
    train_tokens = tokens_from_rows(dataset["train"]["text"])
    vocab = build_vocab(train_tokens, min_freq)
    splits = {"train": torch.tensor(vocab.encode(train_tokens), dtype=torch.long)}
    for name in ("validation", "test"):
        tokens = tokens_from_rows(dataset[name]["text"])
        splits[name] = torch.tensor(vocab.encode(tokens), dtype=torch.long)
    DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
    torch.save({"itos": vocab.itos, "splits": splits, "min_freq": min_freq}, DATA_PATH)
    return {"itos": vocab.itos, "vocab": vocab, "splits": splits, "min_freq": min_freq}


corpus = load_corpus()
vocab = corpus["vocab"]
splits = corpus["splits"]
print(f"vocabulary size: {len(vocab)}")
for name, ids in splits.items():
    unk = int((ids == vocab.unk_id).sum())
    print(f"{name}: {ids.numel()} tokens, {unk} <unk> ({unk / ids.numel():.2%})")

counts = Counter(vocab.itos[index] for index in splits["train"].tolist())
for word, count in counts.most_common(22):
    if word in (UNK, EOS):
        continue
    print(f"{word}\t{count}")


## Batches

The token stream is arranged into columns, one stream per column. Each training window is `SEQ_LEN` tokens, and the target at every position is the next token in that same column. The loss is the cross-entropy of those next-token predictions.


In [ ]:
def batchify(ids, batch_size):
    n_rows = ids.size(0) // batch_size
    trimmed = ids[: n_rows * batch_size]
    return trimmed.view(batch_size, -1).t().contiguous()


def iter_batches(source, seq_len):
    length = source.size(0)
    for start in range(0, length - 1, seq_len):
        window = min(seq_len, length - 1 - start)
        inputs = source[start : start + window]
        targets = source[start + 1 : start + 1 + window]
        yield inputs, targets


train_source = batchify(splits["train"], BATCH_SIZE)
valid_source = batchify(splits["validation"], BATCH_SIZE)
test_source = batchify(splits["test"], BATCH_SIZE)

inputs, targets = next(iter_batches(train_source, SEQ_LEN))
assert inputs.shape == (SEQ_LEN, BATCH_SIZE)
assert torch.equal(inputs[1:, 0], targets[:-1, 0])
print("input ", vocab.decode(inputs[:, 0].tolist()))
print("target", vocab.decode(targets[:, 0].tolist()))


## LSTM

An embedding turns a word id into a vector. The LSTM reads that vector together with its previous hidden state and writes a new hidden state. That state is the memory of the prefix. A linear layer scores every word in the vocabulary.

Between windows the hidden state is detached. Gradients then stop at the window boundary instead of running back through the whole corpus.


In [ ]:
class LSTMLM(nn.Module):
    def __init__(self, vocab_size, embed_size=200, hidden_size=200, num_layers=2, dropout=0.2):
        super().__init__()
        self.drop = nn.Dropout(dropout)
        self.embed = nn.Embedding(vocab_size, embed_size)
        self.lstm = nn.LSTM(
            embed_size,
            hidden_size,
            num_layers=num_layers,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.output = nn.Linear(hidden_size, vocab_size)

    def forward(self, tokens, state=None):
        embedded = self.drop(self.embed(tokens))
        hidden, state = self.lstm(embedded, state)
        logits = self.output(self.drop(hidden))
        return logits, state


def repackage(state):
    if state is None:
        return None
    if isinstance(state, tuple):
        return tuple(repackage(part) for part in state)
    return state.detach()


def run_epoch(model, source, optimizer=None):
    training = optimizer is not None
    model.train(training)
    total_loss = 0.0
    total_tokens = 0
    state = None
    step = 0
    context = torch.enable_grad() if training else torch.no_grad()
    with context:
        for inputs, targets in iter_batches(source, SEQ_LEN):
            inputs = inputs.to(device)
            targets = targets.to(device)
            if training:
                optimizer.zero_grad()
            logits, state = model(inputs, state)
            loss = nn.functional.cross_entropy(logits.reshape(-1, logits.size(-1)), targets.reshape(-1))
            if training:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), CLIP)
                optimizer.step()
            state = repackage(state)
            count = targets.numel()
            total_loss += loss.item() * count
            total_tokens += count
            step += 1
            if training and step % 200 == 0:
                print(f"  batch {step}  loss {loss.item():.3f}  ppl {math.exp(loss.item()):.1f}", flush=True)
    return total_loss / total_tokens


In [ ]:
torch.manual_seed(SEED)
model = LSTMLM(len(vocab)).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)
print(f"parameters: {sum(p.numel() for p in model.parameters()):,}")

config = {
    "model_type": "lstm",
    "embed_size": 200,
    "hidden_size": 200,
    "num_layers": 2,
    "dropout": 0.2,
    "batch_size": BATCH_SIZE,
    "seq_len": SEQ_LEN,
    "lr": LEARNING_RATE,
    "clip": CLIP,
    "min_freq": MIN_FREQ,
    "seed": SEED,
}

best = math.inf
for epoch in range(1, EPOCHS + 1):
    train_loss = run_epoch(model, train_source, optimizer)
    val_loss = run_epoch(model, valid_source)
    train_ppl = math.exp(train_loss)
    val_ppl = math.exp(val_loss)
    print(f"epoch {epoch}/{EPOCHS}  train ppl {train_ppl:.2f}  val ppl {val_ppl:.2f}")
    if val_ppl < best:
        best = val_ppl
        CHECKPOINT_PATH.parent.mkdir(parents=True, exist_ok=True)
        torch.save(
            {
                "model_type": "lstm",
                "state_dict": model.state_dict(),
                "itos": vocab.itos,
                "config": config,
                "val_perplexity": val_ppl,
            },
            CHECKPOINT_PATH,
        )
        print(f"  saved {CHECKPOINT_PATH}")

print(f"best validation perplexity: {best:.2f}")


## Perplexity

Perplexity is `exp(average cross-entropy)`. Lower means the model assigns more probability to the real next words. The score uses the real previous words, the same shift used in training.

The add-1 bigram is the count-based comparison. It only sees one previous word:

`P(word | previous) = (count(previous, word) + 1) / (count(previous) + vocabulary size)`

The first token of a split uses a unigram with the same smoothing. The bigram number is high because one count is added for every vocabulary word.


In [ ]:
def load_checkpoint(path):
    saved = torch.load(path, map_location=device, weights_only=False)
    loaded_vocab = Vocabulary(saved["itos"])
    cfg = saved["config"]
    loaded = LSTMLM(
        len(loaded_vocab),
        embed_size=cfg["embed_size"],
        hidden_size=cfg["hidden_size"],
        num_layers=cfg["num_layers"],
        dropout=cfg["dropout"],
    )
    loaded.load_state_dict(saved["state_dict"])
    loaded.to(device)
    loaded.eval()
    return loaded, loaded_vocab


def bigram_perplexity(train_ids, eval_ids, vocab_size, k=1.0):
    train = train_ids.tolist()
    unigram = Counter(train)
    bigram = Counter(zip(train, train[1:]))
    total = len(train)
    nll = 0.0
    previous = None
    for word in eval_ids.tolist():
        if previous is None:
            probability = (unigram[word] + k) / (total + k * vocab_size)
        else:
            probability = (bigram[(previous, word)] + k) / (unigram[previous] + k * vocab_size)
        nll -= math.log(probability)
        previous = word
    return math.exp(nll / len(eval_ids))


model, vocab = load_checkpoint(CHECKPOINT_PATH)
val_ppl = math.exp(run_epoch(model, valid_source))
test_ppl = math.exp(run_epoch(model, test_source))
bigram_val = bigram_perplexity(splits["train"], splits["validation"], len(vocab))
bigram_test = bigram_perplexity(splits["train"], splits["test"], len(vocab))
print(f"lstm validation perplexity: {val_ppl:.2f}")
print(f"lstm test perplexity: {test_ppl:.2f}")
print(f"bigram validation perplexity: {bigram_val:.2f}")
print(f"bigram test perplexity: {bigram_test:.2f}")


## Generation

The prompt is tokenized with the same rules as the corpus. Unknown words become `<unk>`. Each new word is drawn from the distribution at the last position, then appended, and the model is run again on its own output.

Temperature 0 picks the single most likely word. A higher temperature flattens the distribution, so less likely words are chosen more often. The probability table is computed on the prompt before any new words are added.


In [ ]:
def prompt_ids(text):
    ids = vocab.encode(tokenize_prompt(text))
    return ids or [vocab.unk_id]


def next_logits(ids):
    tokens = torch.tensor(ids, dtype=torch.long, device=device).unsqueeze(1)
    logits, _state = model(tokens)
    return logits[-1, 0]


def sample_id(logits, temperature):
    if temperature <= 0:
        return int(torch.argmax(logits).item())
    probabilities = torch.softmax(logits / temperature, dim=-1)
    return int(torch.multinomial(probabilities, num_samples=1).item())


def next_word_probs(prompt, k=10):
    logits = next_logits(prompt_ids(prompt))
    probabilities = torch.softmax(logits, dim=-1)
    values, indices = torch.topk(probabilities, k)
    return [(vocab.itos[index], probability) for probability, index in zip(values.tolist(), indices.tolist())]


def generate(prompt, n_words, temperature):
    ids = prompt_ids(prompt)
    chosen = []
    with torch.no_grad():
        for _ in range(n_words):
            nxt = sample_id(next_logits(ids), temperature)
            ids.append(nxt)
            chosen.append(nxt)
    return vocab.decode(chosen)


prompt = "The history of science"
for temperature in (0.0, 0.8, 1.2):
    print(f"temperature {temperature}")
    print(generate(prompt, 30, temperature))
    print()

print("next words:")
for word, probability in next_word_probs(prompt):
    print(f"{word}\t{probability:.4f}")


## Gradio

The page asks for a starting phrase, a word count, and a temperature. Generate writes the continuation and fills the table with the 10 most probable next words.


In [ ]:
import gradio as gr


def predict(prompt, n_words, temperature):
    continuation = generate(prompt, int(n_words), float(temperature))
    rows = [[word, round(probability, 6)] for word, probability in next_word_probs(prompt)]
    return continuation, rows


with gr.Blocks(title="WikiText-2 language model") as demo:
    gr.Markdown(
        "Continue a phrase with an LSTM language model trained from scratch on WikiText-2. "
        "The table is the model's distribution for the very next word."
    )
    prompt = gr.Textbox(label="Starting phrase", value="The history of science")
    with gr.Row():
        n_words = gr.Slider(10, 100, value=30, step=1, label="Words to generate")
        temperature = gr.Slider(0, 1.5, value=0.8, step=0.1, label="Temperature (0 picks the most likely word)")
    generate_button = gr.Button("Generate")
    continuation = gr.Textbox(label="Continuation", lines=8)
    table = gr.Dataframe(headers=["word", "probability"], datatype=["str", "number"], type="array", label="Most probable next words")
    generate_button.click(predict, inputs=[prompt, n_words, temperature], outputs=[continuation, table])

demo.launch()
